# Spark Exercises 04 — Window Functions

Window functions are the single most useful Spark skill for real Foundry work, and they are awkward or limited in pandas/Polars. A window lets you compute a value for each row **relative to a group of related rows** — without collapsing the rows like `groupBy` does. Think: *ranking*, *running totals*, *“previous row”*, *“each row’s share of its group”*.

**Data file:** `data/orders.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/orders.csv`, keep only positive `quantity`, and add `revenue = quantity * unit_price`. Call it `o`.

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
o = (orders.filter(F.col("quantity") > 0)
            .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2)))
o.select("order_id", "customer_id", "order_ts", "channel", "revenue").show(5)

+----------+-----------+-------------------+-------+-------+
|  order_id|customer_id|           order_ts|channel|revenue|
+----------+-----------+-------------------+-------+-------+
|ORD-000001|  CUST-0092|2024-07-24 11:01:00|    web|  325.8|
|ORD-000002|  CUST-0016|2024-05-09 14:53:00|    web|  31.46|
|ORD-000003|  CUST-0289|2024-05-12 20:54:00|    web|  10.26|
|ORD-000004|  CUST-0315|2024-07-07 10:26:00|    app| 149.52|
|ORD-000005|  CUST-0356|2024-10-15 08:16:00|  store| 187.91|
+----------+-----------+-------------------+-------+-------+
only showing top 5 rows



### 3. Import the `Window` class.

In [3]:
from pyspark.sql.window import Window

### 4. **Number each customer's orders chronologically.** Define a window partitioned by `customer_id`, ordered by `order_ts`, and add `order_seq = row_number()`. Show `customer_id`, `order_ts`, `order_seq` for 12 rows.

In [4]:
w_time = Window.partitionBy("customer_id").orderBy("order_ts")
o.withColumn("order_seq", F.row_number().over(w_time)) \
 .select("customer_id", "order_ts", "order_seq").show(12)

+-----------+-------------------+---------+
|customer_id|           order_ts|order_seq|
+-----------+-------------------+---------+
|  CUST-0001|2024-01-14 20:34:00|        1|
|  CUST-0001|2024-02-01 18:17:00|        2|
|  CUST-0001|2024-04-17 11:56:00|        3|
|  CUST-0001|2024-05-29 00:41:00|        4|
|  CUST-0001|2024-06-25 03:17:00|        5|
|  CUST-0001|2024-06-27 06:02:00|        6|
|  CUST-0001|2024-07-10 10:25:00|        7|
|  CUST-0001|2024-07-13 21:38:00|        8|
|  CUST-0001|2024-08-10 22:00:00|        9|
|  CUST-0001|2024-09-14 21:02:00|       10|
|  CUST-0001|2024-10-26 16:03:00|       11|
|  CUST-0001|2024-11-06 16:49:00|       12|
+-----------+-------------------+---------+
only showing top 12 rows



### 5. **Top orders per channel.** Define a window partitioned by `channel`, ordered by `revenue` descending, add `rnk = rank()`, and keep only the **top 3** orders in each channel.

In [5]:
w_rev = Window.partitionBy("channel").orderBy(F.col("revenue").desc())
o.withColumn("rnk", F.rank().over(w_rev)) \
 .filter(F.col("rnk") <= 3) \
 .select("channel", "order_id", "revenue", "rnk") \
 .orderBy("channel", "rnk").show(12)

+-------+----------+-------+---+
|channel|  order_id|revenue|rnk|
+-------+----------+-------+---+
|    app|ORD-004003|3031.63|  1|
|    app|ORD-005017|2764.02|  2|
|    app|ORD-001792| 2741.6|  3|
|partner|ORD-004942| 2739.6|  1|
|partner|ORD-000832|2714.58|  2|
|partner|ORD-002120|2563.68|  3|
|  store|ORD-003988|3001.92|  1|
|  store|ORD-004558|2940.96|  2|
|  store|ORD-001431|2853.84|  3|
|    web|ORD-006083|3132.48|  1|
|    web|ORD-003299|3121.44|  2|
|    web|ORD-004511|3016.45|  3|
+-------+----------+-------+---+



### 6. **`rank` vs `dense_rank` — the difference shows up only on ties.** `revenue` is almost unique, so instead rank each **customer's** orders by `quantity` (an integer — lots of ties). Add both `rank()` and `dense_rank()` and show one customer: after a tie, `rank` **skips** the next number(s) while `dense_rank` keeps counting without gaps.

In [6]:
w_qty = Window.partitionBy("customer_id").orderBy(F.col("quantity").desc())
o.withColumn("rnk", F.rank().over(w_qty)) \
 .withColumn("dense", F.dense_rank().over(w_qty)) \
 .filter(F.col("customer_id") == "CUST-0001") \
 .select("order_id", "quantity", "rnk", "dense").show()

+----------+--------+---+-----+
|  order_id|quantity|rnk|dense|
+----------+--------+---+-----+
|ORD-007240|      24|  1|    1|
|ORD-002363|      22|  2|    2|
|ORD-003283|      20|  3|    3|
|ORD-001302|      16|  4|    4|
|ORD-004106|      16|  4|    4|
|ORD-002754|      15|  6|    5|
|ORD-000083|      12|  7|    6|
|ORD-005964|      11|  8|    7|
|ORD-006080|      11|  8|    7|
|ORD-001906|       9| 10|    8|
|ORD-004284|       9| 10|    8|
|ORD-005887|       7| 12|    9|
+----------+--------+---+-----+



### 7. **Running total.** For each customer, compute the cumulative sum of `revenue` over time. Use the `w_time` window with `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

In [7]:
w_run = w_time.rowsBetween(Window.unboundedPreceding, Window.currentRow)
o.withColumn("running_total", F.round(F.sum("revenue").over(w_run), 2)) \
 .select("customer_id", "order_ts", "revenue", "running_total").show(12)

+-----------+-------------------+-------+-------------+
|customer_id|           order_ts|revenue|running_total|
+-----------+-------------------+-------+-------------+
|  CUST-0001|2024-01-14 20:34:00| 153.99|       153.99|
|  CUST-0001|2024-02-01 18:17:00|  99.89|       253.88|
|  CUST-0001|2024-04-17 11:56:00| 127.92|        381.8|
|  CUST-0001|2024-05-29 00:41:00| 808.06|      1189.86|
|  CUST-0001|2024-06-25 03:17:00| 329.12|      1518.98|
|  CUST-0001|2024-06-27 06:02:00|  155.7|      1674.68|
|  CUST-0001|2024-07-10 10:25:00|  12.54|      1687.22|
|  CUST-0001|2024-07-13 21:38:00|  159.5|      1846.72|
|  CUST-0001|2024-08-10 22:00:00| 150.08|       1996.8|
|  CUST-0001|2024-09-14 21:02:00|  173.8|       2170.6|
|  CUST-0001|2024-10-26 16:03:00| 245.85|      2416.45|
|  CUST-0001|2024-11-06 16:49:00| 171.36|      2587.81|
+-----------+-------------------+-------+-------------+
only showing top 12 rows



### 8. **Compare to the previous order.** Add `prev_revenue = lag(revenue)` over `w_time`, and `delta = revenue - prev_revenue`. Show 12 rows.

In [8]:
o.withColumn("prev_revenue", F.lag("revenue").over(w_time)) \
 .withColumn("delta", F.round(F.col("revenue") - F.col("prev_revenue"), 2)) \
 .select("customer_id", "order_ts", "revenue", "prev_revenue", "delta").show(12)

+-----------+-------------------+-------+------------+-------+
|customer_id|           order_ts|revenue|prev_revenue|  delta|
+-----------+-------------------+-------+------------+-------+
|  CUST-0001|2024-01-14 20:34:00| 153.99|        NULL|   NULL|
|  CUST-0001|2024-02-01 18:17:00|  99.89|      153.99|  -54.1|
|  CUST-0001|2024-04-17 11:56:00| 127.92|       99.89|  28.03|
|  CUST-0001|2024-05-29 00:41:00| 808.06|      127.92| 680.14|
|  CUST-0001|2024-06-25 03:17:00| 329.12|      808.06|-478.94|
|  CUST-0001|2024-06-27 06:02:00|  155.7|      329.12|-173.42|
|  CUST-0001|2024-07-10 10:25:00|  12.54|       155.7|-143.16|
|  CUST-0001|2024-07-13 21:38:00|  159.5|       12.54| 146.96|
|  CUST-0001|2024-08-10 22:00:00| 150.08|       159.5|  -9.42|
|  CUST-0001|2024-09-14 21:02:00|  173.8|      150.08|  23.72|
|  CUST-0001|2024-10-26 16:03:00| 245.85|       173.8|  72.05|
|  CUST-0001|2024-11-06 16:49:00| 171.36|      245.85| -74.49|
+-----------+-------------------+-------+------------+-

### 9. **Each order's share of its customer's total.** Define a window partitioned by `customer_id` **without** `orderBy` (so it spans the whole partition), and compute `pct = revenue / sum(revenue) over that window`, as a percentage.

In [9]:
w_cust = Window.partitionBy("customer_id")
o.withColumn("cust_total", F.sum("revenue").over(w_cust)) \
 .withColumn("pct", F.round(F.col("revenue") / F.col("cust_total") * 100, 1)) \
 .select("customer_id", "order_id", "revenue", "cust_total", "pct").show(12)

+-----------+----------+-------+------------------+----+
|customer_id|  order_id|revenue|        cust_total| pct|
+-----------+----------+-------+------------------+----+
|  CUST-0001|ORD-000083| 127.92|2587.8099999999995| 4.9|
|  CUST-0001|ORD-001302| 150.08|2587.8099999999995| 5.8|
|  CUST-0001|ORD-001906| 153.99|2587.8099999999995| 6.0|
|  CUST-0001|ORD-002363| 808.06|2587.8099999999995|31.2|
|  CUST-0001|ORD-002754| 245.85|2587.8099999999995| 9.5|
|  CUST-0001|ORD-003283|  173.8|2587.8099999999995| 6.7|
|  CUST-0001|ORD-004106| 329.12|2587.8099999999995|12.7|
|  CUST-0001|ORD-004284|  155.7|2587.8099999999995| 6.0|
|  CUST-0001|ORD-005887|  99.89|2587.8099999999995| 3.9|
|  CUST-0001|ORD-005964|  159.5|2587.8099999999995| 6.2|
|  CUST-0001|ORD-006080|  12.54|2587.8099999999995| 0.5|
|  CUST-0001|ORD-007240| 171.36|2587.8099999999995| 6.6|
+-----------+----------+-------+------------------+----+
only showing top 12 rows



### 10. **Moving average.** For each customer, compute a 3-order moving average of `revenue` (current row + 2 previous) with `rowsBetween(-2, 0)`.

In [10]:
w_ma = w_time.rowsBetween(-2, 0)
o.withColumn("moving_avg", F.round(F.avg("revenue").over(w_ma), 2)) \
 .select("customer_id", "order_ts", "revenue", "moving_avg").show(12)

+-----------+-------------------+-------+----------+
|customer_id|           order_ts|revenue|moving_avg|
+-----------+-------------------+-------+----------+
|  CUST-0001|2024-01-14 20:34:00| 153.99|    153.99|
|  CUST-0001|2024-02-01 18:17:00|  99.89|    126.94|
|  CUST-0001|2024-04-17 11:56:00| 127.92|    127.27|
|  CUST-0001|2024-05-29 00:41:00| 808.06|    345.29|
|  CUST-0001|2024-06-25 03:17:00| 329.12|     421.7|
|  CUST-0001|2024-06-27 06:02:00|  155.7|    430.96|
|  CUST-0001|2024-07-10 10:25:00|  12.54|    165.79|
|  CUST-0001|2024-07-13 21:38:00|  159.5|    109.25|
|  CUST-0001|2024-08-10 22:00:00| 150.08|    107.37|
|  CUST-0001|2024-09-14 21:02:00|  173.8|    161.13|
|  CUST-0001|2024-10-26 16:03:00| 245.85|    189.91|
|  CUST-0001|2024-11-06 16:49:00| 171.36|     197.0|
+-----------+-------------------+-------+----------+
only showing top 12 rows



### 11. **One row per customer: their biggest order.** Use `row_number()` over `w_rev_cust` (partition by `customer_id`, order by `revenue` desc) and keep only `rn == 1`. Show 10 customers and their top order.

In [11]:
w_rev_cust = Window.partitionBy("customer_id").orderBy(F.col("revenue").desc())
o.withColumn("rn", F.row_number().over(w_rev_cust)) \
 .filter(F.col("rn") == 1) \
 .select("customer_id", "order_id", "revenue").orderBy("customer_id").show(10)

+-----------+----------+-------+
|customer_id|  order_id|revenue|
+-----------+----------+-------+
|  CUST-0001|ORD-002363| 808.06|
|  CUST-0002|ORD-007806| 809.16|
|  CUST-0003|ORD-004550| 592.48|
|  CUST-0004|ORD-002727|2385.26|
|  CUST-0005|ORD-002585|1149.39|
|  CUST-0006|ORD-007992|2241.33|
|  CUST-0007|ORD-003836|2368.52|
|  CUST-0008|ORD-004584| 735.42|
|  CUST-0009|ORD-003334| 711.26|
|  CUST-0010|ORD-006757| 657.73|
+-----------+----------+-------+
only showing top 10 rows



### 12. **Split each customer into spend quartiles.** Add `quartile = ntile(4)` over `w_rev_cust`. Show 12 rows.

In [12]:
o.withColumn("quartile", F.ntile(4).over(w_rev_cust)) \
 .select("customer_id", "order_id", "revenue", "quartile").show(12)

+-----------+----------+-------+--------+
|customer_id|  order_id|revenue|quartile|
+-----------+----------+-------+--------+
|  CUST-0001|ORD-002363| 808.06|       1|
|  CUST-0001|ORD-004106| 329.12|       1|
|  CUST-0001|ORD-002754| 245.85|       1|
|  CUST-0001|ORD-003283|  173.8|       2|
|  CUST-0001|ORD-007240| 171.36|       2|
|  CUST-0001|ORD-005964|  159.5|       2|
|  CUST-0001|ORD-004284|  155.7|       3|
|  CUST-0001|ORD-001906| 153.99|       3|
|  CUST-0001|ORD-001302| 150.08|       3|
|  CUST-0001|ORD-000083| 127.92|       4|
|  CUST-0001|ORD-005887|  99.89|       4|
|  CUST-0001|ORD-006080|  12.54|       4|
+-----------+----------+-------+--------+
only showing top 12 rows

